# LSD Sound-Production Workflow (notebook 07)

End-to-end demonstration of the **LSD sound-production framework** defined in
[`docs/03.md`](../docs/03.md). This notebook:

1. **Trains two different models** on the producer's sound archive (`data/`),
   each with its own `NOISE_INJECT` / `SEED` so the trained decoders differ —
   a non-repeatable, per-run creative axis.
2. Plays each model's **A — baseline output sounds** (sound bank).
3. Uses **B — audio-conditioned variants** to derive novelty from a clip.
4. Uses **C — MIDI-driven synthesis** to play a riff across a bank.
5. Closes the loop with ideas introduced earlier in the progression:
   - **Long-form generation** via overlap-and-add (notebook 02)
   - **BPS modulation + plugin effects** via pedalboard (notebook 03)
   - **Latent-space concatenation** of two model banks into a long piece (notebook 04)

> LSD has no research claim: it is a sound-generation tool. Non-repeatable
> training and noise injection are *features* for artistic exploration.

In [1]:
# --- Generation knobs ---
SEED_A = 3407            # model A seed (set None for non-repeatable)
SEED_B = 1337            # model B seed (different model)
STEPS = 40
TEMPERATURE = 0.9

# --- Archive / training knobs ---
from pathlib import Path

DATA_DIR = Path.cwd().parent / 'data'
AUDIO_LENGTH = 96000        # 4s @ 24kHz
SAMPLE_RATE = 24000
SUBSET_SIZE = 128           # archive subset for fast training
TRAIN_FRAC = 0.8            # leave room for conditioning clips
VAL_FRAC = 0.1
DECODER_EPOCHS = 12
DIFFUSION_EPOCHS = 8
BATCH_SIZE = 8
LR = 1e-3

# --- Per-model artistic axes ---
MODEL_A = dict(noise_std=0.05, label='A·tight')
MODEL_B = dict(noise_std=0.25, label='B·wild')

# --- Inference knobs ---
BANK_SIZE = 6
VARIANTS = 4
CONDITION_STRENGTH = 0.5

# --- Longform / BPS / effects knobs (notebooks 02/03/04) ---
SEGMENT_SECONDS = 8
OVERLAP_FRAC = 0.5
BPS = 3.0
TREMOLO_DEPTH = 0.5
VIBRATO_DEPTH = 1.2
FILTER_SWEEP_MIN = 300
FILTER_SWEEP_MAX = 6000
PB_REVERB_ROOM = 0.35
PB_DELAY_SECONDS = 0.25
PB_DELAY_MIX = 0.25
LATENT_OVERLAP_FRAC = 0.25

LATENT_LENGTH = AUDIO_LENGTH // 320
print(f'Archive: {DATA_DIR}  (subset {SUBSET_SIZE}, latent length {LATENT_LENGTH})')
print(f'Model A: seed={SEED_A}, {MODEL_A}')
print(f'Model B: seed={SEED_B}, {MODEL_B}')

Archive: /Users/tuned-silicon/papers/latent-sound-diffusion/data  (subset 128, latent length 300)
Model A: seed=3407, {'noise_std': 0.05, 'label': 'A·tight'}
Model B: seed=1337, {'noise_std': 0.25, 'label': 'B·wild'}


In [2]:
import math
import random

import numpy as np
import scipy.signal as scisig
import torch
import torch.nn as nn
import torchaudio
import soundfile
from IPython.display import Audio, display
import matplotlib.pyplot as plt
from pedalboard import (
    Pedalboard, Compressor, Reverb, Delay, Gain, Limiter, PeakFilter,
)

from ald_sc.build_prior import build_arrow_prior
from ald_sc.audio_codec import EnCodecEncoder, AudioVAE, BaselineAudioDecoder
from ald_sc.graph_decoder import GraphDecoder
from ald_sc.dit import MinimalDiT
from ald_sc.data import AudioFolderDataset, build_audio_dataloader
from ald_sc.losses import ALDSCLoss
from ald_sc.schedule import CosineSchedule
from ald_sc.trainer import train_audio_decoder, train_audio_diffusion, log_training
from ald_sc.inference import LSDModel

device = torch.device('cpu')
print('Imports OK. Device:', device)

Imports OK. Device: cpu


## Step 1 — Load the sound archive and split

We reserve a held-out slice of the archive for **B** (audio conditioning)
so the producer can pass *unseen* clips to the model.

In [3]:
all_files = sorted(Path(DATA_DIR).glob('*.wav'))
print(f'Archive: {len(all_files)} files in {DATA_DIR}')

random.seed(3407)
selected = random.sample(all_files, min(SUBSET_SIZE, len(all_files)))
random.shuffle(selected)
n = len(selected)
n_train = int(n * TRAIN_FRAC)
n_val = int(n * VAL_FRAC)
train_files = selected[:n_train]
cond_files = selected[n_train:n_train + n_val]   # held-out conditioning clips
test_files = selected[n_train + n_val:]
assert all([train_files, cond_files, test_files]), 'splits must be non-empty'

def make_ds(files):
    ds = AudioFolderDataset(root=str(DATA_DIR), audio_length=AUDIO_LENGTH, sample_rate=SAMPLE_RATE)
    ds.files = files
    return ds

train_ds = make_ds(train_files)
cond_ds = make_ds(cond_files)
print(f'train={len(train_ds)}  cond(to hold out)={len(cond_ds)}  test={len(test_files)}')

Archive: 9811 files in /Users/tuned-silicon/papers/latent-sound-diffusion/data
train=102  cond(to hold out)=12  test=14


## Step 2 — Build the frozen ArrowSpace prior from the training archive

One prior, shared by both models: it encodes the *archive's* feature-space
manifold and is frozen for both decoder trainings.

In [4]:
encoder = EnCodecEncoder(sample_rate=SAMPLE_RATE, bandwidth=24)

train_loader_feat = build_audio_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
features = []
for batch in train_loader_feat:
    z = encoder.extract_features(batch)
    features.append(z.mean(dim=2))
embeddings = torch.cat(features, dim=0)
print(f'Train embeddings: {embeddings.shape}')

Q, K = 8, 4
prior = build_arrow_prior(embeddings, q=Q, k=K)
print(f'Prior: L_F={prior.L_F.shape}, U_q={prior.U_q.shape}, q={prior.q}')

loss_fn = ALDSCLoss(prior=prior, lambda_rec=1.0, lambda_stft=0.0,
                    lambda_chart=0.5, lambda_smooth=0.1)

/Users/tuned-silicon/papers/latent-sound-diffusion/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Train embeddings: torch.Size([96, 128])
Prior: L_F=torch.Size([128, 128]), U_q=torch.Size([128, 8]), q=8


## Step 3 — Train two different models on the same archive

Both models reuse the frozen prior and the same DiT capacity, but differ in:
- the **decoder training seed** (different init / data order), and
- **`noise_std`** latent augmentation (`MODEL_A` tight, `MODEL_B` wild).

This is the core artistic axis of LSD: each run yields a *different* model
and hence a different sound space — by design, not by accident.

In [5]:
def train_one_model(seed: int, noise_std: float, label: str):
    torch.manual_seed(seed)
    decoder = GraphDecoder(128, 1, 128, 32, prior, (2, 4, 5, 8))
    vae = AudioVAE(encoder=encoder, decoder=decoder)
    train_loader = build_audio_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    print(f'Training decoder [{label}]  (seed={seed}, noise_std={noise_std})...')
    list(log_training(
        train_audio_decoder(train_loader, vae, prior, loss_fn,
                            epochs=DECODER_EPOCHS, lr=LR, device=device, noise_std=noise_std),
        label=label,
    ))

    dit = MinimalDiT(latent_channels=128, latent_length=LATENT_LENGTH,
                     patch_size=8, dim=64, depth=2, num_heads=4, spec_dim=3*Q)
    sched = CosineSchedule(num_steps=1000)
    for p in vae.parameters():
        p.requires_grad_(False)
    print(f'Training DiT  [{label}]...')
    list(log_training(
        train_audio_diffusion(train_loader, vae, dit, prior, sched,
                              epochs=DIFFUSION_EPOCHS, lr=LR, device=device),
        label=label + '·DiT',
    ))
    return LSDModel(prior=prior, dit=dit, decoder=decoder, encoder=encoder,
                    schedule=sched, sample_rate=SAMPLE_RATE)

model_a = train_one_model(SEED_A, MODEL_A['noise_std'], MODEL_A['label'])
model_b = train_one_model(SEED_B, MODEL_B['noise_std'], MODEL_B['label'])
print('Two trained models ready: model_a, model_b')

Training decoder [A·tight]  (seed=3407, noise_std=0.05)...
2026-08-02T12:52:50.478909Z [info     ] epoch                          epoch=0 label=A·tight mean_loss=0.20506642883022627 steps=12
2026-08-02T12:52:55.883820Z [info     ] epoch                          epoch=1 label=A·tight mean_loss=0.1681742835789919 steps=12
2026-08-02T12:53:01.396597Z [info     ] epoch                          epoch=2 label=A·tight mean_loss=0.1594178614517053 steps=12
2026-08-02T12:53:07.017977Z [info     ] epoch                          epoch=3 label=A·tight mean_loss=0.16222327450911203 steps=12
2026-08-02T12:53:12.548253Z [info     ] epoch                          epoch=4 label=A·tight mean_loss=0.15375206743677458 steps=12
2026-08-02T12:53:17.999180Z [info     ] epoch                          epoch=5 label=A·tight mean_loss=0.15908811862270036 steps=12
2026-08-02T12:53:23.467487Z [info     ] epoch                          epoch=6 label=A·tight mean_loss=0.16093682497739792 steps=12
2026-08-02T12:53:28

## Step 4 — Mode A: baseline output sounds (sound bank)

Each model produces a *bank* of `BANK_SIZE` baseline sounds. The producer
auditions and selects from these.

In [6]:
bank_a = model_a.generate_sound_bank(n=BANK_SIZE, steps=STEPS, temperature=TEMPERATURE, seed=SEED_A)
bank_b = model_b.generate_sound_bank(n=BANK_SIZE, steps=STEPS, temperature=TEMPERATURE, seed=SEED_B)
print(f'model_a bank: {len(bank_a)} x {bank_a[0].shape}')
print(f'model_b bank: {len(bank_b)} x {bank_b[0].shape}')

print('Model A — bank sound 0:')
display(Audio(bank_a[0].numpy(), rate=SAMPLE_RATE))
print('Model A — bank sound 1:')
display(Audio(bank_a[1].numpy(), rate=SAMPLE_RATE))
print('Model B — bank sound 0 (same slot, different model):')
display(Audio(b_b0 := bank_b[0].numpy(), rate=SAMPLE_RATE))

2026-08-02T12:56:07.014500Z [info     ] sound_bank                     n=6 seed=3407 steps=40 temperature=0.9
2026-08-02T12:56:07.387348Z [info     ] sound_bank                     n=6 seed=1337 steps=40 temperature=0.9
model_a bank: 6 x torch.Size([1, 96000])
model_b bank: 6 x torch.Size([1, 96000])
Model A — bank sound 0:


Model A — bank sound 1:


Model B — bank sound 0 (same slot, different model):


The two models give *different* sounds even at the same bank slot — that's the
artistic lever.

## Step 5 — Mode B: audio-conditioned variants

The producer passes an **existing sound** (here, a held-out archive clip)
and the model generates `VARIANTS` novelty variants of it. `strength`
controls how far from the source the model strays.

In [8]:
cond_clip = train_ds[0]   # (1, T) — a producer sound (Audio wants 1D/2D; condition_on_audio accepts both)
print('Conditioning clip (from archive):')
display(Audio(cond_clip.numpy(), rate=SAMPLE_RATE))

variants_a = model_a.condition_on_audio(cond_clip, n=VARIANTS, steps=STEPS,
                                        strength=CONDITION_STRENGTH, seed=SEED_A)
variants_b = model_b.condition_on_audio(cond_clip, n=VARIANTS, steps=STEPS,
                                        strength=CONDITION_STRENGTH, seed=SEED_B)
print(f'model_a variants: {len(variants_a)}  model_b variants: {len(variants_b)}')

print('Model A — variant 0 of the conditioning clip:')
display(Audio(variants_a[0].numpy(), rate=SAMPLE_RATE))
print('Model B — variant 0 of the same clip:')
display(Audio(variants_b[0].numpy(), rate=SAMPLE_RATE))

Conditioning clip (from archive):


ValueError: Array audio input must be a 1D or 2D array

The variants can themselves be auditioned, selected, fed back into **B**,
or placed in the mix below.

## Step 6 — Mode C: MIDI-driven synthesis

Render a short riff across `model_a`'s bank. The MIDI adapter maps each
note to a pitch-shifted, re-timed bank sound. A full instrument/timbre
model lives in LSD-studio; this is the thin core adapter.

In [ ]:
riff = [
    # (midi_note, start_s, duration_s)
    (60, 0.00, 0.40),
    (63, 0.40, 0.40),
    (67, 0.80, 0.40),
    (72, 1.20, 0.60),
    (67, 1.80, 0.40),
    (63, 2.20, 0.40),
    (60, 2.60, 0.80),
]
render_a = model_a.synthesize_midi(riff, bank=bank_a, pitch_bank_root=60, seed=SEED_A)
render_b = model_b.synthesize_midi(riff, bank=bank_b, pitch_bank_root=60, seed=SEED_B)
print(f'model_a render: {render_a.shape}  ({render_a.shape[-1]/SAMPLE_RATE:.2f}s)')
print('Model A — MIDI render:')
display(Audio(render_a.numpy(), rate=SAMPLE_RATE))
print('Model B — MIDI render of the same riff:')
display(Audio(render_b.numpy(), rate=SAMPLE_RATE))

## Step 7 — Long-form generation (progression: notebook 02)

Stitch the bank into a longer piece via **overlap-and-add** with a Hann
crossfade — the long-form generation technique from notebook 02.

In [ ]:
def overlap_add(segments, overlap_frac, sr=SAMPLE_RATE):
    seg = segments[0]
    L = seg.shape[-1]
    ov = int(L * overlap_frac)
    win = torch.hann_window(2 * ov)
    out = seg.clone()
    for nxt in segments[1:]:
        fade_out = 1 - win[:ov]
        fade_in = win[ov:]
        out[..., -ov:] = out[..., -ov:] * fade_out + nxt[..., :ov] * fade_in
        out = torch.cat([out, nxt[..., ov:]], dim=-1)
    peak = out.abs().max()
    return out / max(peak, 1e-8)

long_a = overlap_add(bank_a, OVERLAP_FRAC)
long_b = overlap_add(bank_b, OVERLAP_FRAC)
# Trim to target length
TARGET = int(SEGMENT_SECONDS * SAMPLE_RATE)
long_a = long_a[..., :TARGET]
long_b = long_b[..., :TARGET]
print(f'long_a: {long_a.shape}  long_b: {long_b.shape}')
print('Model A — longform (overlap-add):')
display(Audio(long_a.numpy(), rate=SAMPLE_RATE))

## Step 8 — BPS modulation + plugin effects (progression: notebook 03)

Shape the long piece with BPS-synchronised **tremolo / vibrato / filter
sweep** LFOs and a **pedalboard** chain (compressor, reverb, delay).

In [ ]:
def make_lfo(num_samples, bps, sr=SAMPLE_RATE, waveform='sine'):
    t = torch.arange(num_samples, dtype=torch.float32) / sr
    phase = 2 * math.pi * bps * t
    return torch.sin(phase)

def apply_tremolo(audio, lfo, depth):
    gain = 1.0 - depth * (1.0 - lfo) / 2.0
    return audio * gain

def apply_vibrato(audio, lfo, depth_semitones, sr=SAMPLE_RATE, window_ms=10):
    if depth_semitones == 0:
        return audio
    win = max(1, int(sr * window_ms / 1000))
    nwin = max(1, len(audio) // win)
    ratio = 2.0 ** (depth_semitones * lfo[:nwin*win].view(nwin, win).mean(1) / 12.0)
    out = torch.zeros_like(audio)
    pos = 0
    for r in ratio:
        span = int(win / r)
        if span <= 0:
            span = 1
        seg = audio[pos:pos+span]
        if seg.numel() == 0:
            break
        tr = torchaudio.transforms.Resample(orig_freq=span, new_freq=win)
        out[pos:pos+win] += tr(seg.unsqueeze(0)).squeeze(0)
        pos += win
    peak = out.abs().max()
    return out / max(peak, 1e-8)

def apply_filter_sweep(audio, lfo, min_hz, max_hz, sr=SAMPLE_RATE, window_ms=25):
    win = max(1, int(sr * window_ms / 1000))
    nwin = max(1, len(audio) // win)
    half = sr / 2
    sig = audio.unsqueeze(0).numpy()
    out = np.zeros_like(sig)
    for i in range(nwin):
        cut = float(min_hz + (max_hz - min_hz) * (lfo[i*win:(i+1)*win].mean().item() + 1) / 2)
        cut = min(max(cut, min_hz), max_hz)
        b, a = scisig.butter(2, cut / half, btype='low')
        out[:, i*win:(i+1)*win] = scisig.lfilter(b, a, sig[:, i*win:(i+1)*win])
    return torch.from_numpy(out).squeeze(0)

lfo = make_lfo(TARGET, BPS)
mod = apply_tremolo(long_a, lfo, TREMOLO_DEPTH)
mod = apply_vibrato(mod, lfo, VIBRATO_DEPTH)
mod = apply_filter_sweep(mod, lfo, FILTER_SWEEP_MIN, FILTER_SWEEP_MAX)

board = Pedalboard([
    Compressor(threshold_db=-20, ratio=4),
    Reverb(room_size=PB_REVERB_ROOM),
    Delay(delay_seconds=PB_DELAY_SECONDS, mix=PB_DELAY_MIX),
    Gain(gain_db=-3),
    Limiter(),
])
final_a = torch.from_numpy(board(mod.unsqueeze(0).numpy(), SAMPLE_RATE)).squeeze(0)
print('Model A — longform after BPS + pedalboard:')
display(Audio(final_a.numpy(), rate=SAMPLE_RATE))

## Step 9 — Latent-space concatenation (progression: notebook 04)

Encode `model_a`'s longform and `model_b`'s longform to their latent
spaces, crossfade in **latent space**, and decode through `model_a`'s
graph decoder. The decoder sees the full concatenated sequence, letting
the project→gate→lift smooth the transition across the two models' banks.

In [ ]:
def to_mono(audio):
    a = audio.squeeze()
    return a if a.dim() == 1 else a.mean(0)

lat_a = encoder.extract_features(long_a.unsqueeze(0).unsqueeze(0))  # (1, 128, Tl)
lat_b = encoder.extract_features(long_b.unsqueeze(0).unsqueeze(0))

def latent_crossfade(z1, z2, overlap_frac):
    L = min(z1.shape[-1], z2.shape[-1])
    ov = int(L * overlap_frac)
    win = torch.hann_window(2 * ov)
    head = z1[..., : L - ov]
    fa = 1 - win[:ov]
    fi = win[ov:]
    cross = z1[..., L - ov:L] * fa + z2[..., :ov] * fi
    tail = z2[..., ov:L]
    return torch.cat([head, cross, tail], dim=-1)

lat_concat = latent_crossfade(lat_a, lat_b, LATENT_OVERLAP_FRAC)
a_concat = lat_concat.mean(dim=2)
c_concat = prior.chart_energy_descriptor(a_concat)
with torch.no_grad():
    concat_audio = model_a.decoder(lat_concat, c_concat)
    concat_audio = concat_audio.squeeze(1)
    pk = concat_audio.abs().max()
    if pk > 0:
        concat_audio = concat_audio / pk

print(f'Latent-space concat: {concat_audio.shape}  ({concat_audio.shape[-1]/SAMPLE_RATE:.2f}s)')
print('Latent-space concatenation of the two model banks:')
display(Audio(concat_audio.numpy(), rate=SAMPLE_RATE))

## Step 10 — Export the produced pieces

Save the baseline bank, the longforms, and the latent-concat piece as
WAV files (a small **results** export). These are the *output sounds* the
producer can now mix, feed back into the model, or hand to a DAW.

In [ ]:
out = Path('results'); out.mkdir(exist_ok=True)

for i, clip in enumerate(bank_a):
    soundfile.write(str(out / f'07_modelA_bank_{i}.wav'), to_mono(clip).numpy(), SAMPLE_RATE)
for i, clip in enumerate(bank_b):
    soundfile.write(str(out / f'07_modelB_bank_{i}.wav'), to_mono(clip).numpy(), SAMPLE_RATE)
soundfile.write(str(out / '07_longformA.wav'), to_mono(long_a).numpy(), SAMPLE_RATE)
soundfile.write(str(out / '07_longformA_effected.wav'), to_mono(final_a).numpy(), SAMPLE_RATE)
soundfile.write(str(out / '07_latent_concat.wav'), to_mono(concat_audio).numpy(), SAMPLE_RATE)
soundfile.write(str(out / '07_midi_renderA.wav'), to_mono(render_a).numpy(), SAMPLE_RATE)

print('Exported to notebooks/results/:')
for p in sorted(out.glob('07_*')):
    print(f'  {p.name}')

## Summary

This notebook exercised the full LSD sound-production workflow end-to-end:

- **Two models** trained on the same archive with different `noise_std`/seed
  → distinct baseline banks (artistic axis).
- **A** sound-bank generation → audition/select.
- **B** audio-conditioned variants → novelty from a held-out clip.
- **C** MIDI-driven synthesis → render a riff across a bank.
- **Long-form** generation (notebook 02): overlap-and-add.
- **BPS modulation + pedalboard** (notebook 03): shape the long piece.
- **Latent-space concatenation** (notebook 04): fuse the two models' banks.

The output sounds live in `notebooks/results/`. They can be:
- mixed into a track,
- fed back to the model (mode B with a bank output as input),
- or concatenated with effects — the loop demonstrated above.